# 📊 Silver Happy — Reporting & Aide à la Décision
### Analyse de l'activité · Dashboards KPI · Insights métier

| | |
|---|---|
| **Données** | 500 interventions · 184 seniors · 5 catégories de service |
| **Objectif** | Suivi d'activité et aide à la décision pour la direction Silver Happy |
| **Stack** | Python · Pandas · Matplotlib · Seaborn |

---
**Parties couvertes :**
- **Partie A** — Description des clients (démographie, abonnements, engagement)
- **Partie B** — Dashboard prestations (performance, CA, évolution temporelle, conversion)
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

BLUE   = '#1A2B49'
LIGHT  = '#7CABD3'
YELLOW = '#FCE297'
GREEN  = '#4CAF50'
RED    = '#E57373'

df = pd.read_csv('../data/dataset_silverhappy.csv', parse_dates=['date_intervention'])
print(f'✅ Dataset chargé : {df.shape[0]} interventions × {df.shape[1]} colonnes')
print(f'   Seniors uniques  : {df["id_senior"].nunique()}')
print(f'   Période          : {df["date_intervention"].min().date()} → {df["date_intervention"].max().date()}')
df.head(3)

---
## Partie A — Description des clients

In [ ]:
# A1 — Profils démographiques
df_clients = df.drop_duplicates(subset=['id_senior'])
print(f'Clients uniques : {len(df_clients)}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('A — Profil démographique des adhérents Silver Happy',
             fontsize=15, fontweight='bold', color=BLUE)

# Genre
genre_counts = df_clients['genre_senior'].value_counts()
axes[0].pie(genre_counts, labels=genre_counts.index,
            autopct='%1.1f%%', colors=[LIGHT, YELLOW],
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Répartition par genre', fontweight='bold')

# Âge
axes[1].hist(df_clients['age_senior'], bins=12, color=BLUE,
             edgecolor='white', linewidth=1)
axes[1].axvline(df_clients['age_senior'].mean(), color=RED,
                linestyle='--', linewidth=2,
                label=f'Moyenne : {df_clients["age_senior"].mean():.1f} ans')
axes[1].set_title('Distribution des âges', fontweight='bold')
axes[1].set_xlabel('Âge')
axes[1].legend()

# Top villes
top_villes = df_clients['ville_senior'].value_counts().head(6)
axes[2].barh(top_villes.index, top_villes.values,
             color=[BLUE if i == 0 else LIGHT for i in range(len(top_villes))])
axes[2].set_title('Top 6 villes', fontweight='bold')
axes[2].set_xlabel('Nombre de seniors')
for i, v in enumerate(top_villes.values):
    axes[2].text(v + 0.2, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/A1_demographics.png', bbox_inches='tight')
plt.show()

In [ ]:
# A2 — Abonnements
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('A — Abonnements clients', fontsize=15, fontweight='bold', color=BLUE)

abo_counts = df_clients['type_abonnement'].value_counts()
colors_abo = [BLUE if a == 'annuel' else LIGHT for a in abo_counts.index]
axes[0].pie(abo_counts, labels=abo_counts.index, autopct='%1.1f%%',
            colors=colors_abo, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Mensuel vs Annuel', fontweight='bold')

# CA par type d'abonnement
ca_abo = df.groupby('type_abonnement')['prix_ttc'].sum().sort_values(ascending=False)
bars = axes[1].bar(ca_abo.index, ca_abo.values,
                   color=[BLUE if a == 'annuel' else LIGHT for a in ca_abo.index],
                   edgecolor='white', linewidth=1)
axes[1].set_title('CA total par type d\'abonnement (€ TTC)', fontweight='bold')
for bar, val in zip(bars, ca_abo.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:,.0f} €', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/A2_abonnements.png', bbox_inches='tight')
plt.show()

In [ ]:
# A3 — Ancienneté et engagement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('A — Ancienneté et engagement des adhérents',
             fontsize=15, fontweight='bold', color=BLUE)

axes[0].hist(df_clients['anciennete_senior_j'], bins=10,
             color=YELLOW, edgecolor=BLUE, linewidth=1)
axes[0].set_title('Ancienneté des clients (jours)', fontweight='bold')
axes[0].set_xlabel('Jours depuis l\'inscription')

axes[1].scatter(df_clients['anciennete_senior_j'],
                df_clients['nb_connexions_senior'],
                alpha=0.6, color=BLUE, s=40)
axes[1].set_title('Ancienneté vs Connexions', fontweight='bold')
axes[1].set_xlabel('Ancienneté (jours)')
axes[1].set_ylabel('Nb connexions')

corr = df_clients[['anciennete_senior_j','nb_connexions_senior']].corr().iloc[0,1]
axes[1].annotate(f'r = {corr:.2f}', xy=(0.05, 0.92),
                 xycoords='axes fraction', fontsize=11, color=RED)

plt.tight_layout()
plt.savefig('../outputs/A3_engagement.png', bbox_inches='tight')
plt.show()

---
## Partie B — Dashboard des prestations

In [ ]:
# B1 — KPI par service
df_done = df[df['statut_intervention'] == 'terminee'].copy()

dashboard = df_done.groupby('nom_service').agg(
    nb_interventions = ('id_intervention', 'count'),
    clients_uniques  = ('id_senior',       'nunique'),
    ca_genere        = ('prix_ttc',        'sum'),
    commission_sh    = ('commission_sh',   'sum'),
    note_moyenne     = ('note_avis',       'mean'),
    duree_moy        = ('duree_heures',    'mean'),
).round(2).sort_values('ca_genere', ascending=False)

print('📋 Dashboard KPI — Prestations terminées :')
print(dashboard.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('B — Performance des prestations (interventions terminées)',
             fontsize=15, fontweight='bold', color=BLUE)

top_ca = dashboard['ca_genere'].sort_values(ascending=True)
colors = [GREEN if i == len(top_ca)-1 else BLUE if i >= len(top_ca)-3 else LIGHT
          for i in range(len(top_ca))]
top_ca.plot(kind='barh', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('CA généré par service (€ TTC)', fontweight='bold')
axes[0].set_xlabel('Chiffre d\'affaires (€)')
for i, val in enumerate(top_ca.values):
    axes[0].text(val + 20, i, f'{val:,.0f}€', va='center', fontsize=9)

note_service = dashboard['note_moyenne'].sort_values(ascending=True).dropna()
note_service.plot(kind='barh', ax=axes[1], color=LIGHT, edgecolor='white')
axes[1].set_title('Note moyenne par service (/5)', fontweight='bold')
axes[1].set_xlim(0, 5.5)
axes[1].axvline(4.0, color=GREEN, linestyle='--', alpha=0.6, label='Seuil 4.0')
axes[1].legend()
for i, val in enumerate(note_service.values):
    axes[1].text(val + 0.05, i, f'{val:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/B1_kpi_services.png', bbox_inches='tight')
plt.show()

In [ ]:
# B2 — Évolution temporelle du CA
df['mois_annee'] = df['date_intervention'].dt.to_period('M')
evolution = df.groupby('mois_annee').agg(
    nb_interventions = ('id_intervention', 'count'),
    ca_mensuel       = ('prix_ttc',        'sum'),
    commission_mois  = ('commission_sh',   'sum'),
).reset_index()
evolution['mois_annee'] = evolution['mois_annee'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.suptitle('B — Évolution mensuelle de l\'activité',
             fontsize=15, fontweight='bold', color=BLUE)

axes[0].plot(evolution['mois_annee'], evolution['ca_mensuel'],
             color=BLUE, linewidth=2.5, marker='o', markersize=6)
axes[0].fill_between(evolution['mois_annee'], evolution['ca_mensuel'],
                     alpha=0.15, color=BLUE)
axes[0].set_title('CA mensuel (€ TTC)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(evolution['mois_annee'], evolution['nb_interventions'],
            color=LIGHT, edgecolor='white', linewidth=1)
axes[1].set_title('Nombre d\'interventions par mois', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../outputs/B2_evolution.png', bbox_inches='tight')
plt.show()

In [ ]:
# B3 — Taux de conversion et statuts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('B — Taux de conversion et statuts des demandes',
             fontsize=15, fontweight='bold', color=BLUE)

statut_d = df['statut_demande'].value_counts()
colors_sd = {'accepte': GREEN, 'refuse': RED, 'annulee': YELLOW, 'en_attente': LIGHT}
pie_colors = [colors_sd.get(s, BLUE) for s in statut_d.index]
axes[0].pie(statut_d, labels=statut_d.index, autopct='%1.1f%%',
            colors=pie_colors, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Statut des demandes', fontweight='bold')

statut_i = df['statut_intervention'].value_counts()
colors_si = {'terminee': GREEN, 'annulee': RED, 'en_cours': YELLOW, 'planifiee': LIGHT}
pie_colors_i = [colors_si.get(s, BLUE) for s in statut_i.index]
axes[1].pie(statut_i, labels=statut_i.index, autopct='%1.1f%%',
            colors=pie_colors_i, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Statut des interventions', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/B3_conversion.png', bbox_inches='tight')
plt.show()

tx_conversion = (statut_d.get('accepte', 0) / statut_d.sum()) * 100
print(f'\n📊 Taux de conversion (demandes acceptées) : {tx_conversion:.1f}%')

In [ ]:
# Synthèse exécutive
total_ca       = df[df['statut_intervention']=='terminee']['prix_ttc'].sum()
total_comm     = df[df['statut_intervention']=='terminee']['commission_sh'].sum()
note_globale   = df['note_avis'].mean()
tx_annuel      = (df_clients['type_abonnement']=='annuel').mean() * 100

print('=' * 55)
print('       SYNTHÈSE EXÉCUTIVE — SILVER HAPPY')
print('=' * 55)
print(f'  Adhérents actifs      : {len(df_clients)}')
print(f'  Interventions totales : {len(df)}')
print(f'  CA total (terminées)  : {total_ca:,.0f} €')
print(f'  Commission SH         : {total_comm:,.0f} €')
print(f'  Note de satisfaction  : {note_globale:.2f} / 5')
print(f'  Taux abonnés annuels  : {tx_annuel:.1f}%')
print('=' * 55)